# Ability dimensions

Reads the `keys4to8` CNRM fits on accuracy. The axes are fixed only up to an orthogonal map,
so only lengths and angles of alpha are compared, never its components.


In [ ]:
import jax
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from omegaconf import OmegaConf
from scipy.linalg import orthogonal_procrustes

from vsrgrating.config import CONFIG_DIR
from vsrgrating.criteria import readouts, reproducibility
from vsrgrating.data import load_split, locate_dataset, subset
from vsrgrating.evaluation import Fitted, load_run, locate_run
from vsrgrating.irt.cnrm.model import ItemParams
from vsrgrating.readout import cnrm

jax.config.update("jax_enable_x64", True)

ROOT = CONFIG_DIR.parent
DATASET_ROOT = ROOT / "data" / "datasets"
RUNS_ROOT = ROOT / "runs"
DATASET = "keys4to8"
RESPONSE = "acc"
FITS = {2: "keys4to8_cnrm_d2", 3: "keys4to8_cnrm_d3"}
CRITERIA = "keys4to8_criteria_d2"
Y = 0.95
EXPERIMENT_5 = {"4K-7K": 0.716, "4K-6K": 0.773}

In [ ]:
dataset = locate_dataset(DATASET_ROOT, DATASET)
keys = pl.read_parquet(dataset / "items.parquet").sort("item_idx")["keys"].to_numpy()
train = load_split(dataset, RESPONSE, "train")
fits = {
	dims: load_run(name, locate_run(RUNS_ROOT, name, RESPONSE, dataset.name))
	for dims, name in FITS.items()
}

In [ ]:
def alpha_of(fit: Fitted) -> np.ndarray:
	"""The fit's alpha as (items, D), D = 1 included."""
	item = fit.item
	assert isinstance(item, ItemParams)
	return np.asarray(item.alpha).reshape(keys.size, -1)


def mode_cosines(alpha: np.ndarray, keep: np.ndarray) -> dict[str, float]:
	"""Cosine between each pair of key modes' mean alpha: the ability correlation it implies."""
	modes = np.unique(keys[keep])
	mean = {k: alpha[keep & (keys == k)].mean(axis=0) for k in modes}
	unit = {k: v / np.linalg.norm(v) for k, v in mean.items()}
	return {
		f"{a}K-{b}K": float(unit[a] @ unit[b]) for i, a in enumerate(modes) for b in modes[i + 1 :]
	}


def aligned_angles(left: np.ndarray, right: np.ndarray) -> np.ndarray:
	"""Each chart's angle in degrees between 2 fits, once right is rotated onto left."""
	# N(0, I) is unchanged by any orthogonal map, so 2 fits' axes differ by one.
	rotation, _ = orthogonal_procrustes(right, left)
	turned = right @ rotation
	norms = np.linalg.norm(left, axis=1) * np.linalg.norm(turned, axis=1)
	return np.degrees(np.arccos(np.clip((left * turned).sum(axis=1) / norms, -1.0, 1.0)))

## Implied correlation between key modes

Experiment 5 measured 4K-7K and 4K-6K directly, on the people who play both.


In [ ]:
everything = np.ones(keys.size, dtype=bool)
cosines = {f"D={dims}": mode_cosines(alpha_of(fit), everything) for dims, fit in fits.items()}
pairs = list(next(iter(cosines.values())))
pl.DataFrame(
	{
		"pair": pairs,
		**{name: [values[p] for p in pairs] for name, values in cosines.items()},
		"experiment 5": [EXPERIMENT_5.get(p) for p in pairs],
	}
)

## Stability over the Q1 halves


In [ ]:
criteria = OmegaConf.load(RUNS_ROOT / CRITERIA / RESPONSE / "config.yaml")
seeds = range(criteria.q1.seed, criteria.q1.seed + criteria.q1.repeats)


def half(seed: int, side: str) -> np.ndarray:
	path = locate_run(RUNS_ROOT, CRITERIA, f"{RESPONSE}-q1-{seed}-{side}", dataset.name)
	return alpha_of(load_run(side, path))


def both_seen(seed: int) -> np.ndarray:
	first, second = reproducibility.halves(train.n_persons, seed)
	left = readouts.seen(subset(train, first[train.person]))
	return left & readouts.seen(subset(train, second[train.person]))


halves = {seed: (both_seen(seed), half(seed, "a"), half(seed, "b")) for seed in seeds}
spread = pl.DataFrame(
	[
		{"seed": seed, "half": side, **mode_cosines(alpha, seen)}
		for seed, (seen, a, b) in halves.items()
		for side, alpha in (("a", a), ("b", b))
	]
)
spread.select(pl.exclude("seed", "half")).describe()

In [ ]:
angles = pl.concat(
	pl.DataFrame({"seed": seed, "keys": keys[seen], "angle": aligned_angles(a[seen], b[seen])})
	for seed, (seen, a, b) in halves.items()
)
angles.group_by("keys").agg(
	pl.len().alias("pairs"),
	pl.col("angle").median().alias("median"),
	pl.col("angle").quantile(0.9).alias("p90"),
).sort("keys")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k in np.unique(keys):
	chosen_angles = angles.filter(pl.col("keys") == k)["angle"].to_numpy()
	ax.hist(chosen_angles, bins=90, range=(0, 180), histtype="step", density=True, label=f"{k}K")

ax.set_xlabel("angle between halves (degrees)")
ax.legend()

plt.show()

## Length of alpha and d by key mode

On the fit the criteria were run on.


In [ ]:
chosen = load_run("chosen", locate_run(RUNS_ROOT, criteria.base, RESPONSE, dataset.name))
item = chosen.item

assert isinstance(item, ItemParams)
assert chosen.model.transform is not None and chosen.raw is not None

charts = pl.DataFrame(
	{
		"keys": keys,
		"norm": np.linalg.norm(alpha_of(chosen), axis=1),
		"d": cnrm.difficulty(item, chosen.model.transform, chosen.raw, Y),
	}
)
quantiles = (0.05, 0.25, 0.5, 0.75, 0.95)
charts.group_by("keys").agg(
	pl.len().alias("charts"),
	*(pl.col(c).quantile(q).alias(f"{c} q{q:.2f}") for c in ("norm", "d") for q in quantiles),
).sort("keys")

In [ ]:
modes = np.unique(keys)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, column, label in zip(axes, ("norm", "d"), ("|alpha|", f"d({Y})"), strict=True):
	groups = [charts.filter(pl.col("keys") == k)[column].to_numpy() for k in modes]
	ax.boxplot(groups, tick_labels=[f"{k}K" for k in modes])
	ax.set_ylabel(label)

plt.show()